# Hybrid matcher runtime

Полный код runtime-слоя ML-модели из файла `hybrid_matcher_runtime.py`.

Этот слой используется backend-приложением: он загружает артефакты, подготавливает запросы поставщика и возвращает top-3 релевантных лота для каждого товара.


## ImportFrom: ImportFrom


In [ ]:
from __future__ import annotations

import argparse

import json

import pickle

from pathlib import Path

from typing import Any

import numpy as np

import pandas as pd

from scipy.sparse import load_npz

import ml_baseline as base

SELLER_COLUMN_ALIASES = {
    "Наименвоание": "Наименование",
    "Наименвоание ": "Наименование",
    "Наименование ": "Наименование",
}

def normalize_seller_frame(df: pd.DataFrame) -> pd.DataFrame:
    normalized = df.rename(columns=SELLER_COLUMN_ALIASES).copy()
    required = ["id", "Категория", "Наименование", "Характеристики"]
    missing = [column for column in required if column not in normalized.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return normalized[required].fillna("")

def read_seller_csv(path: str | Path) -> pd.DataFrame:
    csv_path = Path(path)
    last_error: Exception | None = None
    for separator in (";", ","):
        try:
            df = pd.read_csv(csv_path, sep=separator, encoding="utf-8-sig", dtype=str)
            return normalize_seller_frame(df)
        except Exception as exc:  # pragma: no cover - fallback path
            last_error = exc
    if last_error is not None:
        raise last_error
    raise ValueError(f"Failed to read seller CSV: {csv_path}")

def build_short_explanation(match: dict[str, Any]) -> str:
    parts: list[str] = []
    matched_unit_name = str(match.get("matched_unit_name", "") or "").strip()
    matched_terms = [term for term in str(match.get("matched_terms", "")).split(", ") if term]
    type_relation = str(match.get("type_relation", "") or "")
    confidence = str(match.get("confidence", "") or "")
    warnings: list[str] = []

    if matched_unit_name:
        parts.append(f'В лоте есть позиция "{matched_unit_name}"')
    else:
        parts.append("Лот выбран по общему совпадению текста")

    if matched_terms:
        parts.append(f"совпали слова: {', '.join(matched_terms[:4])}")

    if type_relation == "exact":
        parts.append("тип товара совпадает")
    elif type_relation == "compatible":
        parts.append("тип товара близкий")
    elif type_relation == "mismatch":
        warnings.append("тип товара выглядит спорно")

    if float(match.get("service_penalty", 0.0) or 0.0) > 0:
        warnings.append("лот похож на сервисный или ремонтный")

    if not bool(match.get("has_expected_type_signal", True)):
        warnings.append("внутри лота слабый сигнал по типу товара")

    if confidence:
        parts.append(f"уверенность {confidence}")

    if warnings:
        parts.append("предупреждения: " + "; ".join(warnings))

    return ". ".join(parts).strip()


## ClassDef: HybridLotMatcher


In [ ]:
class HybridLotMatcher:
    def __init__(self, package_root: str | Path):
        self.package_root = Path(package_root).resolve()
        self.config = json.loads((self.package_root / "config" / "model_config.json").read_text(encoding="utf-8"))

        bundle_path = self.package_root / self.config["artifacts"]["runtime_bundle"]
        with bundle_path.open("rb") as fh:
            bundle = pickle.load(fh)

        self.bundle = bundle
        self.purchases = bundle["purchases"]
        self.lot_ids = bundle["lot_ids"]
        self.lot_to_indices = bundle["lot_to_indices"]
        self.purchase_product_type = bundle["purchase_product_type"]
        self.purchase_service = bundle["purchase_service"]
        self.purchase_tokens = bundle["purchase_tokens"]
        self.purchase_type_signal_text = bundle["purchase_type_signal_text"]

        artifacts_root = self.package_root / "artifacts" / "index"
        with (artifacts_root / "word_vectorizer.pkl").open("rb") as fh:
            self.word_vectorizer = pickle.load(fh)

        self.char_vectorizer = None
        char_vectorizer_path = artifacts_root / "char_vectorizer.pkl"
        if char_vectorizer_path.exists():
            with char_vectorizer_path.open("rb") as fh:
                self.char_vectorizer = pickle.load(fh)

        self.x_lot_docs = load_npz(artifacts_root / "lot_docs.npz").tocsr()
        self.x_item_docs = load_npz(artifacts_root / "item_docs.npz").tocsr()

        self.svd = None
        self.lot_embeddings = None
        svd_path = artifacts_root / "svd.pkl"
        lot_embeddings_path = artifacts_root / "lot_embeddings.npy"
        if svd_path.exists() and lot_embeddings_path.exists():
            with svd_path.open("rb") as fh:
                self.svd = pickle.load(fh)
            self.lot_embeddings = np.load(lot_embeddings_path)

        self.params = self.config["scoring"]

    def _read_seller_input(self, seller_input: str | Path | pd.DataFrame) -> pd.DataFrame:
        if isinstance(seller_input, pd.DataFrame):
            return normalize_seller_frame(seller_input)
        return read_seller_csv(Path(seller_input))

    def _prepare_seller_queries(self, seller: pd.DataFrame) -> dict[str, Any]:
        seller_name = seller["Наименование"].map(base.normalize_text).fillna("")
        seller_category = seller["Категория"].map(base.normalize_category).fillna("")
        seller_chars = seller["Характеристики"].map(base.parse_characteristics).fillna("")
        seller_text = seller_name + " " + seller_name + " " + seller_name + " " + seller_category + " " + seller_chars
        seller_product_type = np.array(
            [
                base.resolve_seller_product_type(name, category, chars)
                for name, chars, category in zip(
                    seller_name.tolist(),
                    seller_chars.tolist(),
                    seller_category.tolist(),
                )
            ],
            dtype=object,
        )
        seller_service = [base.has_service_signal(text) for text in (seller_name + " " + seller_category).tolist()]
        seller_tokens = [base.significant_tokens(text) for text in seller_text.tolist()]
        q_docs = base.transform_with_vectorizers(seller_text, self.word_vectorizer, self.char_vectorizer, self._args())
        q_embeddings = None
        if self.svd is not None and self.lot_embeddings is not None:
            q_embeddings = base.normalize_dense_rows(self.svd.transform(q_docs).astype(np.float32, copy=False))
        return {
            "seller": seller,
            "seller_text": seller_text,
            "seller_product_type": seller_product_type,
            "seller_service": seller_service,
            "seller_tokens": seller_tokens,
            "q_docs": q_docs,
            "q_embeddings": q_embeddings,
        }

    def _args(self) -> argparse.Namespace:
        args = base.parse_args([])
        args.char_weight = self.params["char_weight"]
        args.top_rows = self.params["top_rows"]
        args.top_lots = self.params["top_lots"]
        args.semantic_top_lots = self.params["semantic_top_lots"]
        args.chunk_size = self.params["chunk_size"]
        args.embedding_weight = self.params["embedding_weight"]
        args.item_presence_weight = self.params["item_presence_weight"]
        args.type_exact_bonus = self.params["type_exact_bonus"]
        args.type_compatible_bonus = self.params["type_compatible_bonus"]
        args.type_mismatch_penalty = self.params["type_mismatch_penalty"]
        args.type_missing_keyword_penalty = self.params["type_missing_keyword_penalty"]
        args.overlap_bonus_weight = self.params["overlap_bonus_weight"]
        args.service_penalty = self.params["service_penalty"]
        return args

    def predict(self, seller_input: str | Path | pd.DataFrame, top_lots: int | None = None) -> pd.DataFrame:
        args = self._args()
        if top_lots is not None:
            args.top_lots = int(top_lots)

        prepared = self._prepare_seller_queries(self._read_seller_input(seller_input))
        seller = prepared["seller"]
        q_docs = prepared["q_docs"]
        q_embeddings = prepared["q_embeddings"]
        seller_tokens = prepared["seller_tokens"]
        seller_service = prepared["seller_service"]
        seller_product_type = prepared["seller_product_type"]

        rows: list[dict[str, Any]] = []

        for chunk_start in range(0, q_docs.shape[0], args.chunk_size):
            chunk_end = min(q_docs.shape[0], chunk_start + args.chunk_size)
            lot_scores = q_docs[chunk_start:chunk_end] @ self.x_lot_docs.T
            semantic_scores = None
            if q_embeddings is not None and self.lot_embeddings is not None:
                semantic_scores = (q_embeddings[chunk_start:chunk_end] @ self.lot_embeddings.T).astype(np.float32, copy=False)

            for local_idx, product_idx in enumerate(range(chunk_start, chunk_end)):
                query_row = lot_scores.getrow(local_idx)
                lexical_candidates = base.topk_sparse_row(query_row, args.top_rows)
                semantic_candidates = (
                    base.topk_dense_scores(semantic_scores[local_idx], args.semantic_top_lots)
                    if semantic_scores is not None
                    else []
                )
                candidate_lots: dict[int, dict[str, float]] = {}
                for lot_idx, score in lexical_candidates:
                    current = candidate_lots.setdefault(int(lot_idx), {"lexical_score": 0.0, "semantic_score": 0.0})
                    current["lexical_score"] = max(current["lexical_score"], float(score))
                for lot_idx, score in semantic_candidates:
                    current = candidate_lots.setdefault(int(lot_idx), {"lexical_score": 0.0, "semantic_score": 0.0})
                    current["semantic_score"] = max(current["semantic_score"], float(score))

                query_tokens = seller_tokens[product_idx]
                query_is_service = seller_service[product_idx]
                query_type = str(seller_product_type[product_idx])
                per_lot: list[dict[str, Any]] = []

                for lot_idx, candidate_scores in candidate_lots.items():
                    lot_id = self.lot_ids[lot_idx]
                    lot_indices = np.asarray(self.lot_to_indices.get(lot_id, []), dtype=np.int32)
                    if lot_indices.size == 0:
                        continue

                    item_similarity = (q_docs[product_idx] @ self.x_item_docs[lot_indices].T).toarray().ravel()
                    if item_similarity.size == 0:
                        continue

                    best_item_pos = int(item_similarity.argmax())
                    purchase_idx = int(lot_indices[best_item_pos])
                    lexical_score = float(candidate_scores["lexical_score"])
                    semantic_score = float(candidate_scores["semantic_score"])
                    item_presence_score = float(item_similarity[best_item_pos])
                    retrieval_score = (1.0 - args.embedding_weight) * lexical_score + args.embedding_weight * semantic_score
                    text_score = (
                        args.item_presence_weight * item_presence_score
                        + (1.0 - args.item_presence_weight) * retrieval_score
                    )

                    overlap = 0.0
                    if query_tokens:
                        overlap = len(query_tokens & self.purchase_tokens[purchase_idx]) / len(query_tokens)
                    overlap_bonus = args.overlap_bonus_weight * min(overlap, 0.8)

                    service_penalty = 0.0
                    if not query_is_service and self.purchase_service[purchase_idx]:
                        service_penalty = args.service_penalty

                    doc_type = str(self.purchase_product_type[purchase_idx])
                    type_score, type_relation = base.type_adjustment(
                        seller_type=query_type,
                        purchase_type=doc_type,
                        exact_bonus=args.type_exact_bonus,
                        compatible_bonus=args.type_compatible_bonus,
                        mismatch_penalty=args.type_mismatch_penalty,
                    )
                    expected_type_signal = base.has_expected_type_signal(
                        seller_type=query_type,
                        purchase_type=doc_type,
                        purchase_text=self.purchase_type_signal_text[purchase_idx],
                    )
                    type_missing_penalty = 0.0 if expected_type_signal else args.type_missing_keyword_penalty
                    final_score = text_score + overlap_bonus + type_score - service_penalty - type_missing_penalty

                    matched_terms = sorted(query_tokens & self.purchase_tokens[purchase_idx])
                    purchase_row = self.purchases.iloc[purchase_idx]
                    per_lot.append(
                        {
                            "seller_id": seller.at[product_idx, "id"],
                            "seller_category": seller.at[product_idx, "Категория"],
                            "seller_name": seller.at[product_idx, "Наименование"],
                            "pn_lot": purchase_row["pn_lot"],
                            "platform_brief": purchase_row["platform_brief"],
                            "publish_date": purchase_row["publish_date"],
                            "platform_number": purchase_row["platform_number"],
                            "lot_number": purchase_row["lot_number"],
                            "procedure_name": purchase_row["procedure_name"],
                            "lot_subject": purchase_row["lot_subject"],
                            "matched_unit_name": purchase_row["unit_name"],
                            "unit_okpd_code": purchase_row["unit_okpd_code"],
                            "seller_product_type": query_type,
                            "matched_product_type": doc_type,
                            "type_relation": type_relation,
                            "lexical_score": lexical_score,
                            "semantic_score": semantic_score,
                            "retrieval_score": retrieval_score,
                            "item_presence_score": item_presence_score,
                            "text_score": text_score,
                            "overlap_bonus": overlap_bonus,
                            "type_score": type_score,
                            "has_expected_type_signal": expected_type_signal,
                            "type_missing_penalty": type_missing_penalty,
                            "service_penalty": service_penalty,
                            "final_score": final_score,
                            "score_100": base.clipped_score_100(final_score),
                            "confidence": base.confidence_label(final_score),
                            "matched_terms": ", ".join(matched_terms[:20]),
                        }
                    )

                ranked = sorted(per_lot, key=lambda item: item["final_score"], reverse=True)[: args.top_lots]
                for rank, match in enumerate(ranked, start=1):
                    match["lot_rank"] = rank
                    match["explanation_short"] = build_short_explanation(match)
                    rows.append(match)

        return pd.DataFrame(rows)


## FunctionDef: cli


In [ ]:
def cli() -> None:
    parser = argparse.ArgumentParser(description="Hybrid lot matcher runtime")
    parser.add_argument("--package-root", default=Path(__file__).resolve().parent, help="Path to package root")
    parser.add_argument("--input-csv", required=True, help="Buyer matrix CSV")
    parser.add_argument("--output-csv", required=True, help="Output CSV with top lots")
    parser.add_argument("--top-lots", type=int, default=3, help="Returned lots per buyer product")
    args = parser.parse_args()

    matcher = HybridLotMatcher(args.package_root)
    result = matcher.predict(args.input_csv, top_lots=args.top_lots)
    result.to_csv(args.output_csv, index=False, encoding="utf-8-sig")
    print(f"Saved {len(result)} rows to {args.output_csv}")

if __name__ == "__main__":
    cli()
